In [27]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

In [28]:
# ============================================================
# 2. LOAD DATA
# ============================================================

player_stats = pd.read_csv("player_match_stats_with_positions.csv")
events = pd.read_csv("events.csv")

print("Player stats:", player_stats.shape)
print("Events:", events.shape)

player_stats.head()

/var/folders/l1/c11rhp6x6pl9nqfm7t9hdxmw0000gn/T/ipykernel_43136/4070624082.py:6: DtypeWarning: Columns (0,1,2,3,4,5,12,14,15,16,19,20,22,27,32,37,40,42,43,44,45,46,47,48,49,51,52,53,55,56,57,58,59,76,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118) have mixed types. Specify dtype option on import or set low_memory=False.
  events = pd.read_csv("events.csv")


Player stats: (5336, 28)
Events: (598221, 119)


,competition_name,season_name,competition_short,tournament_id,match_id,team,player,total_events,shots,passes,...,completed_passes,pass_completion_pct,position,position_id,position_x,position_y,jersey_number,starter,from,to
0,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Franck Yannick Kessié,134,1,39,...,38,97.435897,Right Center Midfield,13.0,60.0,52.0,8,True,00:00,NaN
1,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Ghislain Konan,230,0,72,...,59,81.944444,Left Back,6.0,28.0,12.0,3,True,00:00,NaN
2,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Ibrahim Sangaré,210,0,66,...,57,86.363636,Center Defensive Midfield,10.0,45.0,40.0,18,True,00:00,89:38
3,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Idrissa Doumbia,11,0,4,...,3,75.000000,Center Defensive Midfield,10.0,45.0,40.0,25,False,89:38,NaN
4,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,Jean-Philippe Krasso,83,4,10,...,5,50.000000,Center Forward,23.0,105.0,40.0,11,True,00:00,60:47


In [29]:
# ============================================================
# 3. CREATE PLAYER MASTER BASE
# ============================================================

player_master = (
    player_stats
    .groupby([
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team",
        "player"
    ])
    .agg(
        matches=("match_id", "nunique"),
        total_events=("total_events", "sum"),
        shots=("shots", "sum"),
        passes=("passes", "sum"),
        completed_passes=("completed_passes", "sum"),
        carries=("carries", "sum"),
        pressures=("pressures", "sum"),
        recoveries=("recoveries", "sum"),
        duels=("duels", "sum"),
        interceptions=("interceptions", "sum"),
        blocks=("blocks", "sum"),
        clearances=("clearances", "sum"),
        xg=("xg", "sum"),
        position=("position", lambda x: x.dropna().mode()[0] if len(x.dropna()) > 0 else np.nan),
        position_x=("position_x", lambda x: x.dropna().mode()[0] if len(x.dropna()) > 0 else np.nan),
        position_y=("position_y", lambda x: x.dropna().mode()[0] if len(x.dropna()) > 0 else np.nan),
        starter_matches=("starter", "sum")
    )
    .reset_index()
)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,recoveries,duels,interceptions,blocks,clearances,xg,position,position_x,position_y,starter_matches
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,4,3,0,0,0,0.036418,Right Wing,88.0,68.0,1
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,13,0,0,0,0,0.000000,Goalkeeper,8.0,40.0,3
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,8,4,0,8,7,0.466483,Right Center Back,25.0,52.0,3
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,6,6,0,4,3,1.890892,Center Forward,105.0,40.0,3
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,3,1,0,1,0,0.109249,Left Wing,60.0,12.0,1


In [30]:
# ============================================================
# 4. CREATE PER MATCH METRICS
# ============================================================

volume_cols = [
    "total_events",
    "shots",
    "passes",
    "completed_passes",
    "carries",
    "pressures",
    "recoveries",
    "duels",
    "interceptions",
    "blocks",
    "clearances",
    "xg"
]

for col in volume_cols:
    player_master[f"{col}_per_match"] = (
        player_master[col] / player_master["matches"]
    )

player_master["pass_completion_pct"] = np.where(
    player_master["passes"] > 0,
    player_master["completed_passes"] / player_master["passes"] * 100,
    0
)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,completed_passes_per_match,carries_per_match,pressures_per_match,recoveries_per_match,duels_per_match,interceptions_per_match,blocks_per_match,clearances_per_match,xg_per_match,pass_completion_pct
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,9.333333,19.000000,3.000000,1.333333,1.000000,0.0,0.000000,0.000000,0.012139,58.333333
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,21.333333,13.333333,0.000000,4.333333,0.000000,0.0,0.000000,0.000000,0.000000,83.116883
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,67.000000,55.666667,2.666667,2.666667,1.333333,0.0,2.666667,2.333333,0.155494,94.366197
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,6.666667,9.000000,14.000000,2.000000,2.000000,0.0,1.333333,1.000000,0.630297,62.500000
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,26.500000,26.500000,7.000000,1.500000,0.500000,0.0,0.500000,0.000000,0.054625,76.811594


In [31]:
# ============================================================
# 5. CREATE PLAYER PROFILE METRICS
# ============================================================

player_master["attacking_volume"] = (
    player_master["xg_per_match"] * 0.6 +
    player_master["shots_per_match"] * 0.4
)

player_master["passing_volume"] = (
    player_master["passes_per_match"] * 0.6 +
    player_master["completed_passes_per_match"] * 0.4
)

player_master["carrying_volume"] = (
    player_master["carries_per_match"]
)

player_master["pressing_volume"] = (
    player_master["pressures_per_match"]
)

player_master["ball_winning_volume"] = (
    player_master["recoveries_per_match"] +
    player_master["interceptions_per_match"] +
    player_master["duels_per_match"]
)

player_master["defensive_volume"] = (
    player_master["pressures_per_match"] +
    player_master["recoveries_per_match"] +
    player_master["interceptions_per_match"] +
    player_master["blocks_per_match"] +
    player_master["clearances_per_match"]
)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,blocks_per_match,clearances_per_match,xg_per_match,pass_completion_pct,attacking_volume,passing_volume,carrying_volume,pressing_volume,ball_winning_volume,defensive_volume
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,0.000000,0.000000,0.012139,58.333333,0.273950,13.333333,19.000000,3.000000,2.333333,4.333333
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,0.000000,0.000000,0.000000,83.116883,0.000000,23.933333,13.333333,0.000000,4.333333,4.333333
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,2.666667,2.333333,0.155494,94.366197,0.493297,69.400000,55.666667,2.666667,4.000000,10.333333
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,1.333333,1.000000,0.630297,62.500000,1.578178,9.066667,9.000000,14.000000,4.000000,18.333333
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,0.500000,0.000000,0.054625,76.811594,0.632775,31.300000,26.500000,7.000000,2.000000,9.000000


In [32]:
# ============================================================
# 6. CREATE TOURNAMENT PERCENTILES
# ============================================================

rank_metrics = [
    "total_events_per_match",
    "xg_per_match",
    "shots_per_match",
    "passes_per_match",
    "completed_passes_per_match",
    "pass_completion_pct",
    "carries_per_match",
    "pressures_per_match",
    "recoveries_per_match",
    "duels_per_match",
    "interceptions_per_match",
    "blocks_per_match",
    "clearances_per_match",
    "attacking_volume",
    "passing_volume",
    "carrying_volume",
    "pressing_volume",
    "ball_winning_volume",
    "defensive_volume"
]

for metric in rank_metrics:
    player_master[f"{metric}_rank"] = (
        player_master
        .groupby("tournament_id")[metric]
        .rank(pct=True)
    )

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,duels_per_match_rank,interceptions_per_match_rank,blocks_per_match_rank,clearances_per_match_rank,attacking_volume_rank,passing_volume_rank,carrying_volume_rank,pressing_volume_rank,ball_winning_volume_rank,defensive_volume_rank
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,0.313877,0.245595,0.138767,0.190529,0.537445,0.354626,0.524229,0.180617,0.190529,0.126652
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,0.088106,0.245595,0.138767,0.190529,0.200441,0.566079,0.385463,0.030837,0.441630,0.126652
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,0.418502,0.245595,0.888767,0.786344,0.755507,0.993392,0.993392,0.160793,0.396476,0.355727
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,0.579295,0.245595,0.676211,0.607930,0.988987,0.257709,0.253304,0.791850,0.396476,0.670705
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,0.193833,0.245595,0.342511,0.190529,0.806167,0.689427,0.720264,0.405286,0.157489,0.288546


In [33]:
# ============================================================

# 7. POSITION GROUPS + ROLE CLASSIFICATION

# ============================================================

def get_position_group(position):

    position = str(position)

    if position == "Goalkeeper":

        return "Goalkeeper"

    elif any(x in position for x in ["Back", "Center Back", "Wing Back"]):

        return "Defender"

    elif any(x in position for x in ["Midfield"]):

        return "Midfielder"

    elif any(x in position for x in ["Wing", "Forward", "Striker"]):

        return "Forward"

    else:

        return "Other"

player_master["position_group"] = player_master["position"].apply(get_position_group)

def classify_general_role(row):

    if row["position_group"] == "Goalkeeper":

        return "Goalkeeper"

    if row["attacking_volume_rank"] >= 0.80:

        return "Box Threat"

    elif row["passing_volume_rank"] >= 0.80 and row["pass_completion_pct_rank"] >= 0.60:

        return "Tempo Controller"

    elif row["carrying_volume_rank"] >= 0.80:

        return "Progressive Carrier"

    elif row["pressing_volume_rank"] >= 0.80:

        return "Pressing Player"

    elif row["ball_winning_volume_rank"] >= 0.80:

        return "Ball Winner"

    else:

        return "Balanced Role"

def classify_offensive_role(row):

    if row["position_group"] == "Goalkeeper":

        return "Build-Up Goalkeeper"

    if row["xg_per_match_rank"] >= 0.80 or row["shots_per_match_rank"] >= 0.80:

        return "Finisher / Shot Threat"

    elif row["passes_per_match_rank"] >= 0.80:

        return "High-Volume Passer"

    elif row["carries_per_match_rank"] >= 0.80:

        return "Ball Carrier"

    else:

        return "Balanced Offensive Role"

def classify_defensive_role(row):

    if row["position_group"] == "Goalkeeper":

        return "Goal Prevention"

    if row["pressures_per_match_rank"] >= 0.80:

        return "Pressing Specialist"

    elif row["ball_winning_volume_rank"] >= 0.80:

        return "Ball Winner"

    elif row["defensive_volume_rank"] >= 0.80:

        return "Defensive Contributor"

    else:

        return "Balanced Defensive Role"

player_master["general_role"] = player_master.apply(classify_general_role, axis=1)

player_master["offensive_role"] = player_master.apply(classify_offensive_role, axis=1)

player_master["defensive_role"] = player_master.apply(classify_defensive_role, axis=1)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,attacking_volume_rank,passing_volume_rank,carrying_volume_rank,pressing_volume_rank,ball_winning_volume_rank,defensive_volume_rank,position_group,general_role,offensive_role,defensive_role
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,0.537445,0.354626,0.524229,0.180617,0.190529,0.126652,Forward,Balanced Role,Balanced Offensive Role,Balanced Defensive Role
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,0.200441,0.566079,0.385463,0.030837,0.441630,0.126652,Goalkeeper,Goalkeeper,Build-Up Goalkeeper,Goal Prevention
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,0.755507,0.993392,0.993392,0.160793,0.396476,0.355727,Defender,Tempo Controller,High-Volume Passer,Balanced Defensive Role
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,0.988987,0.257709,0.253304,0.791850,0.396476,0.670705,Forward,Box Threat,Finisher / Shot Threat,Balanced Defensive Role
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,0.806167,0.689427,0.720264,0.405286,0.157489,0.288546,Forward,Box Threat,Finisher / Shot Threat,Balanced Defensive Role


In [34]:
# ============================================================

# 8. CREATE PLAYER IMPORTANCE SCORE — POSITION ADJUSTED

# ============================================================

player_master["field_importance_score"] = (

    player_master["total_events_per_match_rank"] * 0.20 +

    player_master["attacking_volume_rank"] * 0.20 +

    player_master["passing_volume_rank"] * 0.20 +

    player_master["carrying_volume_rank"] * 0.15 +

    player_master["pressing_volume_rank"] * 0.15 +

    player_master["ball_winning_volume_rank"] * 0.10

) * 100

player_master["gk_importance_score"] = (

    player_master["total_events_per_match_rank"] * 0.25 +

    player_master["passing_volume_rank"] * 0.25 +

    player_master["pass_completion_pct_rank"] * 0.20 +

    player_master["recoveries_per_match_rank"] * 0.10 +

    player_master["clearances_per_match_rank"] * 0.10 +

    player_master["blocks_per_match_rank"] * 0.10

) * 100

player_master["importance_score"] = np.where(

    player_master["position_group"] == "Goalkeeper",

    player_master["gk_importance_score"],

    player_master["field_importance_score"]

)

def importance_tier(score):

    if score >= 80:

        return "Key Player"

    elif score >= 65:

        return "Important Player"

    elif score >= 50:

        return "Rotation / Contributor"

    else:

        return "Low Usage / Limited Impact"

player_master["importance_tier"] = player_master["importance_score"].apply(

    importance_tier

)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,ball_winning_volume_rank,defensive_volume_rank,position_group,general_role,offensive_role,defensive_role,field_importance_score,gk_importance_score,importance_score,importance_tier
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,0.190529,0.126652,Forward,Balanced Role,Balanced Offensive Role,Balanced Defensive Role,37.676211,26.266520,37.676211,Low Usage / Limited Impact
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,0.441630,0.126652,Goalkeeper,Goalkeeper,Build-Up Goalkeeper,Goal Prevention,33.171806,47.742291,47.742291,Low Usage / Limited Impact
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,0.396476,0.355727,Defender,Tempo Controller,High-Volume Passer,Balanced Defensive Role,75.947137,90.220264,75.947137,Important Player
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,0.396476,0.670705,Forward,Box Threat,Finisher / Shot Threat,Balanced Defensive Role,53.122247,37.125551,53.122247,Rotation / Contributor
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,0.157489,0.288546,Forward,Box Threat,Finisher / Shot Threat,Balanced Defensive Role,60.572687,49.823789,60.572687,Rotation / Contributor


In [35]:
# ============================================================

# 9. CREATE PLAYER STRENGTHS AND WEAKNESSES

# ============================================================

def get_player_strengths(row):

    strengths = []

    if row["position_group"] == "Goalkeeper":

        if row["passing_volume_rank"] >= 0.80:

            strengths.append("High involvement in build-up")

        if row["pass_completion_pct_rank"] >= 0.80:

            strengths.append("Strong passing accuracy")

        if row["clearances_per_match_rank"] >= 0.80:

            strengths.append("High clearance activity")

        if row["blocks_per_match_rank"] >= 0.80:

            strengths.append("Strong shot-stopping involvement")

        if row["recoveries_per_match_rank"] >= 0.80:

            strengths.append("Active in recoveries")

        if len(strengths) == 0:

            strengths.append("Balanced goalkeeper profile")

        return strengths

    if row["xg_per_match_rank"] >= 0.80:

        strengths.append("Strong xG generation")

    if row["shots_per_match_rank"] >= 0.80:

        strengths.append("High shot volume")

    if row["passes_per_match_rank"] >= 0.80:

        strengths.append("High passing involvement")

    if row["pass_completion_pct_rank"] >= 0.80:

        strengths.append("Strong passing accuracy")

    if row["carries_per_match_rank"] >= 0.80:

        strengths.append("High carry volume")

    if row["pressures_per_match_rank"] >= 0.80:

        strengths.append("High pressing activity")

    if row["ball_winning_volume_rank"] >= 0.80:

        strengths.append("Strong ball-winning activity")

    if len(strengths) == 0:

        strengths.append("Balanced profile without one dominant strength")

    return strengths

def get_player_weaknesses(row):

    weaknesses = []

    if row["position_group"] == "Goalkeeper":

        if row["passing_volume_rank"] <= 0.20:

            weaknesses.append("Low involvement in build-up")

        if row["pass_completion_pct_rank"] <= 0.20:

            weaknesses.append("Low passing accuracy")

        if row["clearances_per_match_rank"] <= 0.20:

            weaknesses.append("Low clearance activity")

        if row["blocks_per_match_rank"] <= 0.20:

            weaknesses.append("Limited shot-stopping event volume")

        if len(weaknesses) == 0:

            weaknesses.append("No clear major weakness from selected goalkeeper indicators")

        return weaknesses

    if row["xg_per_match_rank"] <= 0.20:

        weaknesses.append("Limited xG generation")

    if row["shots_per_match_rank"] <= 0.20:

        weaknesses.append("Low shot volume")

    if row["passes_per_match_rank"] <= 0.20:

        weaknesses.append("Low passing involvement")

    if row["carries_per_match_rank"] <= 0.20:

        weaknesses.append("Low carry volume")

    if row["pressures_per_match_rank"] <= 0.20:

        weaknesses.append("Low pressing activity")

    if row["ball_winning_volume_rank"] <= 0.20:

        weaknesses.append("Limited ball-winning activity")

    if len(weaknesses) == 0:

        weaknesses.append("No clear major weakness from selected indicators")

    return weaknesses

player_master["strengths"] = player_master.apply(get_player_strengths, axis=1)

player_master["weaknesses"] = player_master.apply(get_player_weaknesses, axis=1)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,position_group,general_role,offensive_role,defensive_role,field_importance_score,gk_importance_score,importance_score,importance_tier,strengths,weaknesses
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,Forward,Balanced Role,Balanced Offensive Role,Balanced Defensive Role,37.676211,26.266520,37.676211,Low Usage / Limited Impact,[Balanced profile without one dominant strength],"[Low pressing activity, Limited ball-winning a..."
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,Goalkeeper,Goalkeeper,Build-Up Goalkeeper,Goal Prevention,33.171806,47.742291,47.742291,Low Usage / Limited Impact,[Active in recoveries],"[Low clearance activity, Limited shot-stopping..."
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,Defender,Tempo Controller,High-Volume Passer,Balanced Defensive Role,75.947137,90.220264,75.947137,Important Player,"[High passing involvement, Strong passing accu...",[Low pressing activity]
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,Forward,Box Threat,Finisher / Shot Threat,Balanced Defensive Role,53.122247,37.125551,53.122247,Rotation / Contributor,"[Strong xG generation, High shot volume]",[No clear major weakness from selected indicat...
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,Forward,Box Threat,Finisher / Shot Threat,Balanced Defensive Role,60.572687,49.823789,60.572687,Rotation / Contributor,[High shot volume],[Limited ball-winning activity]


In [36]:
# ============================================================
# 10. CREATE ROLE FIT SCORES
# ============================================================

target_roles = {
    "Box Threat": {
        "attacking_volume_rank": 0.55,
        "xg_per_match_rank": 0.25,
        "shots_per_match_rank": 0.20
    },
    "Tempo Controller": {
        "passing_volume_rank": 0.45,
        "pass_completion_pct_rank": 0.30,
        "passes_per_match_rank": 0.25
    },
    "Progressive Carrier": {
        "carrying_volume_rank": 0.70,
        "carries_per_match_rank": 0.30
    },
    "Pressing Player": {
        "pressing_volume_rank": 0.70,
        "pressures_per_match_rank": 0.30
    },
    "Ball Winner": {
        "ball_winning_volume_rank": 0.70,
        "recoveries_per_match_rank": 0.20,
        "interceptions_per_match_rank": 0.10
    },
    "Balanced Role": {
        "attacking_volume_rank": 0.20,
        "passing_volume_rank": 0.20,
        "carrying_volume_rank": 0.15,
        "pressing_volume_rank": 0.20,
        "ball_winning_volume_rank": 0.25
    }
}

role_fit_rows = []

for _, row in player_master.iterrows():

    for target_role, weights in target_roles.items():

        score = 0

        for metric, weight in weights.items():
            score += row.get(metric, 0) * weight

        role_fit_rows.append({
            "competition_name": row["competition_name"],
            "season_name": row["season_name"],
            "competition_short": row["competition_short"],
            "tournament_id": row["tournament_id"],
            "team": row["team"],
            "player": row["player"],
            "position": row["position"],
            "current_role": row["general_role"],
            "target_role": target_role,
            "role_fit_score": score * 100
        })

role_fit = pd.DataFrame(role_fit_rows)

role_fit["role_fit_rank"] = (
    role_fit
    .groupby(["tournament_id", "team", "player"])["role_fit_score"]
    .rank(method="first", ascending=False)
)

role_fit.head()

,competition_name,season_name,competition_short,tournament_id,team,player,position,current_role,target_role,role_fit_score,role_fit_rank
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,Balanced Role,Box Threat,52.929515,1.0
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,Balanced Role,Tempo Controller,28.617841,4.0
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,Balanced Role,Progressive Carrier,52.422907,2.0
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,Balanced Role,Pressing Player,18.061674,6.0
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,Right Wing,Balanced Role,Ball Winner,21.255507,5.0


In [37]:
# ============================================================
# 11. ADD BEST ROLE FITS TO PLAYER MASTER
# ============================================================

best_roles = (
    role_fit[
        role_fit["role_fit_rank"] <= 3
    ]
    .sort_values(["tournament_id", "team", "player", "role_fit_rank"])
    .groupby(["tournament_id", "team", "player"])
    .agg(
        best_role_fits=("target_role", lambda x: ", ".join(x.astype(str))),
        best_role_fit_scores=("role_fit_score", lambda x: ", ".join([f"{v:.1f}" for v in x]))
    )
    .reset_index()
)

player_master = player_master.merge(
    best_roles,
    on=["tournament_id", "team", "player"],
    how="left"
)

player_master.head()

,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,offensive_role,defensive_role,field_importance_score,gk_importance_score,importance_score,importance_tier,strengths,weaknesses,best_role_fits,best_role_fit_scores
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,Balanced Offensive Role,Balanced Defensive Role,37.676211,26.266520,37.676211,Low Usage / Limited Impact,[Balanced profile without one dominant strength],"[Low pressing activity, Limited ball-winning a...","Box Threat, Progressive Carrier, Balanced Role","52.9, 52.4, 34.1"
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,Build-Up Goalkeeper,Goal Prevention,33.171806,47.742291,47.742291,Low Usage / Limited Impact,[Active in recoveries],"[Low clearance activity, Limited shot-stopping...","Tempo Controller, Ball Winner, Progressive Car...","59.3, 49.5, 38.5"
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,High-Volume Passer,Balanced Defensive Role,75.947137,90.220264,75.947137,Important Player,"[High passing involvement, Strong passing accu...",[Low pressing activity],"Progressive Carrier, Tempo Controller, Box Threat","99.3, 97.3, 75.4"
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,Finisher / Shot Threat,Balanced Defensive Role,53.122247,37.125551,53.122247,Rotation / Contributor,"[Strong xG generation, High shot volume]",[No clear major weakness from selected indicat...,"Box Threat, Pressing Player, Balanced Role","98.2, 79.2, 54.5"
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,Finisher / Shot Threat,Balanced Defensive Role,60.572687,49.823789,60.572687,Rotation / Contributor,[High shot volume],[Limited ball-winning activity],"Box Threat, Progressive Carrier, Tempo Controller","77.2, 72.0, 61.6"


In [38]:
# ============================================================
# 12. ROUND AND SAVE PLAYER MASTER
# ============================================================

numeric_cols = player_master.select_dtypes(include=np.number).columns

player_master[numeric_cols] = player_master[numeric_cols].round(2)
role_fit["role_fit_score"] = role_fit["role_fit_score"].round(2)

player_master.to_csv("player_master.csv", index=False)
role_fit.to_csv("player_role_fit.csv", index=False)

print("player_master.csv saved successfully!")
print("player_role_fit.csv saved successfully!")

print(player_master.shape)
player_master.head()

player_master.csv saved successfully!
player_role_fit.csv saved successfully!
(1937, 73)


,competition_name,season_name,competition_short,tournament_id,team,player,matches,total_events,shots,passes,...,offensive_role,defensive_role,field_importance_score,gk_importance_score,importance_score,importance_tier,strengths,weaknesses,best_role_fits,best_role_fit_scores
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Adam Ounas,3,215,2,48,...,Balanced Offensive Role,Balanced Defensive Role,37.68,26.27,37.68,Low Usage / Limited Impact,[Balanced profile without one dominant strength],"[Low pressing activity, Limited ball-winning a...","Box Threat, Progressive Carrier, Balanced Role","52.9, 52.4, 34.1"
1,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Anthony Louis Mandréa,3,211,0,77,...,Build-Up Goalkeeper,Goal Prevention,33.17,47.74,47.74,Low Usage / Limited Impact,[Active in recoveries],"[Low clearance activity, Limited shot-stopping...","Tempo Controller, Ball Winner, Progressive Car...","59.3, 49.5, 38.5"
2,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Aïssa Mandi,3,620,3,213,...,High-Volume Passer,Balanced Defensive Role,75.95,90.22,75.95,Important Player,"[High passing involvement, Strong passing accu...",[Low pressing activity],"Progressive Carrier, Tempo Controller, Box Threat","99.3, 97.3, 75.4"
3,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Baghdad Bounedjah,3,235,9,32,...,Finisher / Shot Threat,Balanced Defensive Role,53.12,37.13,53.12,Rotation / Contributor,"[Strong xG generation, High shot volume]",[No clear major weakness from selected indicat...,"Box Threat, Pressing Player, Balanced Role","98.2, 79.2, 54.5"
4,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,Fares Chaïbi,2,208,3,69,...,Finisher / Shot Threat,Balanced Defensive Role,60.57,49.82,60.57,Rotation / Contributor,[High shot volume],[Limited ball-winning activity],"Box Threat, Progressive Carrier, Tempo Controller","77.2, 72.0, 61.6"
